# Tutorial: Datasets and Saving Strategies

The following demonstrates how to use the `ep.saving_strategies` to save data in different formats `ep.DataSet` to load the saved data.

We will use the loaded variables from the cdf file to showcase how to save them with different formats. We will add additional variables so that we have more to save.


In [1]:
from datetime import datetime, timezone

from astropy import units as u

import el_paso as ep

ep.setup_logging()

extraction_infos = [
    ep.ExtractionInfo(
        result_key="Epoch",
        name_or_column="Epoch_Ele",
        unit=ep.units.cdf_epoch,
    ),
    ep.ExtractionInfo(
        result_key="FEDU",
        name_or_column="FEDU",
        unit=(u.cm**2 * u.s * u.sr * u.keV) ** (-1),
    ),
    ep.ExtractionInfo(
        result_key="xGEO",
        name_or_column="Position_Ele",
        unit=u.km,
    ),
]

start_time = datetime(2017, 7, 14, tzinfo=timezone.utc)
end_time = datetime(2017, 7, 14, 23, 59, 59, tzinfo=timezone.utc)

file_name_stem = "rbspa_rel04_ect-hope-pa-l3_YYYYMMDD_.{6}.cdf"

ep.download(
    start_time,
    end_time,
    save_path=".",
    download_url="https://spdf.gsfc.nasa.gov/pub/data/rbsp/rbspa/l3/ect/hope/pitchangle/rel04/YYYY/",
    file_name_stem=file_name_stem,
    file_cadence="daily",
    method="request",
    skip_existing=True,
)

variables = ep.extract_variables_from_files(
    start_time, end_time, "daily", data_path=".", file_name_stem=file_name_stem, extraction_infos=extraction_infos
)
variables

[INFO    ] 2026-06-15 09:58:11 - el_paso.download:272 - Error downloading file from https://spdf.gsfc.nasa.gov/pub/data/rbsp/rbspa/l3/ect/hope/pitchangle/rel04/2017/: HTTPSConnectionPool(host='spdf.gsfc.nasa.gov', port=443): Read timed out. (read timeout=10)
[INFO    ] 2026-06-15 09:58:11 - el_paso.download:32 - download finished in 10.816 seconds
[INFO    ] 2026-06-15 09:58:11 - el_paso.extract_variables_from_files:112 - Extracting variables ...


{'Epoch': Variable holding (3940,) data points with metadata: VariableMetadata(unit=Unit("cdf_epoch"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], description='', processing_notes='', standard_name=''),
 'FEDU': Variable holding (3940, 11, 72) data points with metadata: VariableMetadata(unit=Unit("1 / (keV s sr cm2)"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], description='', processing_notes='', standard_name=''),
 'xGEO': Variable holding (3940, 3) data points with metadata: VariableMetadata(unit=Unit("km"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], description='', processing_notes='', standard_name='')}

## Single file strategy

First, we want to save the variables using the `SingleFileStrategy`. This is the simplest way to save variables, as everything is simple put into one file. Dependent on the file ending, different file formats will be saved. Possible formats are ".mat", ".nc", ".cdf" and ".h5".

The units of the variables are not changed when using the `SingleFileStrategy`.


In [6]:
saving_strategy = ep.saving_strategies.SingleFileStrategy("rbsp_hope_example.mat")
ep.save(
    variables,
    saving_strategy=saving_strategy,
    start_time=start_time,
    end_time=end_time,
    time_var=variables["Epoch"],
    ignore_validation=True,
)

[INFO    ] 2026-06-15 10:00:56 - el_paso.saving_strategies.single_file_strategy:294 - Saving file rbsp_hope_example.mat...
[INFO    ] 2026-06-15 10:00:56 - el_paso.save:2 - save finished in 0.488 seconds


Let's inspect what got saved. The variables are turned into simple numpy arrays before saving. You can see that also a _metadata_ variable has been saved. We will look closer at metadata in a different tutorial.


In [7]:
import scipy.io as sio

with open("rbsp_hope_example.mat", "rb") as f:
    loaded_data = sio.loadmat(f)
    print("Keys: ", loaded_data.keys())
    print("Metadata: ", loaded_data["metadata"])
    print("xGEO[0,:]: ", loaded_data["xGEO"][0, :] * u.Unit(loaded_data["metadata"]["xGEO"][0, 0]["unit"][0, 0][0]))

Keys:  dict_keys(['__header__', '__version__', '__globals__', 'Epoch', 'FEDU', 'xGEO', 'metadata'])
Metadata:  [[(array([[(array(['cdf_epoch'], dtype='<U9'), array([[0]]), array(['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], dtype='<U46'), array([], dtype='<U1'), array([], dtype='<U1'), array(['Epoch'], dtype='<U5'))]],
        dtype=[('unit', 'O'), ('original_cadence_seconds', 'O'), ('source_files', 'O'), ('description', 'O'), ('processing_notes', 'O'), ('standard_name', 'O')]), array([[(array(['1 / (keV s sr cm2)'], dtype='<U18'), array([[0]]), array(['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], dtype='<U46'), array([], dtype='<U1'), array([], dtype='<U1'), array(['FEDU'], dtype='<U4'))]],
        dtype=[('unit', 'O'), ('original_cadence_seconds', 'O'), ('source_files', 'O'), ('description', 'O'), ('processing_notes', 'O'), ('standard_name', 'O')]), array([[(array(['km'], dtype='<U2'), array([[0]]), array(['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], dtype='<U46'), a

# Saving strategies

## GFZStrategy

All data at GFZ is stored under a standard, which we call `GFZStandard` and is saved using `GFZStrategy` where multiple monthly files are generated for different variables. The standard inlcudes name of the variables that are saved. All files are saved as `.mat` files. By using the `GFZStrategy`, the variables are automatically sorted into the corresponding files, based on the key of the variable_dict. Additionally, the variables are converted into the units as described in the standard.

## MonthlyRBStrategy

Under this strategy, the data is stored in monthly files and all the variables are written (appended) to one monthly file. In this strategy, different standards (`GFZStandard` and `PRBEMStandard`) can be used to define the name of the variables.

While using these strategies, we have to store the variables in a dictionary with certain keys, as specified by the `InternalName` type alias. These variable name and the corresponding value in the variables dictionary are statically typechecked and are also validated internally in order to refrain users from saving incorrect key and value types. We can ignore this validation by setting `ignore_validation=True` in `ep.save(..., ignore_validation=True)`


In [8]:
from el_paso.typing import InternalName
from datetime import datetime, timezone

from astropy import units as u

import el_paso as ep


ep.setup_logging()

extraction_infos = [
    ep.ExtractionInfo(
        result_key="Epoch",
        name_or_column="Epoch_Ele",
        unit=ep.units.cdf_epoch,
    ),
    ep.ExtractionInfo(
        result_key="xGEO",
        name_or_column="Position_Ele",
        unit=u.km,
    ),
]

start_time = datetime(2017, 7, 14, tzinfo=timezone.utc)
end_time = datetime(2017, 7, 14, 23, 59, 59, tzinfo=timezone.utc)

file_name_stem = "rbspa_rel04_ect-hope-pa-l3_YYYYMMDD_.{6}.cdf"

ep.download(
    start_time,
    end_time,
    save_path=".",
    download_url="https://spdf.gsfc.nasa.gov/pub/data/rbsp/rbspa/l3/ect/hope/pitchangle/rel04/YYYY/",
    file_name_stem=file_name_stem,
    file_cadence="daily",
    method="request",
    skip_existing=True,
)

variables = ep.extract_variables_from_files(
    start_time, end_time, "daily", data_path=".", file_name_stem=file_name_stem, extraction_infos=extraction_infos
)
variables

[INFO    ] 2026-06-15 10:01:12 - el_paso.download:272 - Error downloading file from https://spdf.gsfc.nasa.gov/pub/data/rbsp/rbspa/l3/ect/hope/pitchangle/rel04/2017/: HTTPSConnectionPool(host='spdf.gsfc.nasa.gov', port=443): Read timed out. (read timeout=10)
[INFO    ] 2026-06-15 10:01:12 - el_paso.download:29 - download finished in 10.388 seconds
[INFO    ] 2026-06-15 10:01:12 - el_paso.extract_variables_from_files:112 - Extracting variables ...


{'Epoch': Variable holding (3940,) data points with metadata: VariableMetadata(unit=Unit("cdf_epoch"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], description='', processing_notes='', standard_name=''),
 'xGEO': Variable holding (3940, 3) data points with metadata: VariableMetadata(unit=Unit("km"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf'], description='', processing_notes='', standard_name='')}

In [9]:
variables_to_save: dict[InternalName, ep.Variable] = {
    "Epoch": variables["Epoch"],
    "Position": variables["xGEO"],
}


mrb_gfz = ep.saving_strategies.MonthlyRBStrategy(
    "./RBSP/mrb_gfz",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

gfz_gfz = ep.saving_strategies.GFZStrategy(
    "./RBSP/gfz_gfz",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

gfz_prbem = ep.saving_strategies.GFZStrategy(
    "./RBSP/gfz_prbem",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.PRBEMStandard(),
)

mrb_prbem = ep.saving_strategies.MonthlyRBStrategy(
    "./RBSP/mrb_prbem",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.PRBEMStandard(),
)


ep.save(variables_to_save, mrb_gfz, start_time, end_time, time_var=variables["Epoch"])
ep.save(variables_to_save, gfz_gfz, start_time, end_time, time_var=variables["Epoch"])
ep.save(variables_to_save, mrb_prbem, start_time, end_time, time_var=variables["Epoch"])
ep.save(variables_to_save, gfz_prbem, start_time, end_time, time_var=variables["Epoch"])

[WARNING ] 2026-06-15 10:01:15 - el_paso.saving_strategy:79 - Could not find target variable(s) Alpha, Alpha_Eq, B_Calc, B_Eq, Energy_FEDU, FEDU, InvK, InvMu, L_m, L_star, MLT, PSD, R_Eq!
[INFO    ] 2026-06-15 10:01:15 - el_paso.saving_strategy:183 - Saving file: /home/bhaas/Repos/EL_PASO/bunch_download/tutorials/RBSP/mrb_gfz/RBSP/rbspa/rbspa_hope_20170701to20170731_T89.nc
[INFO    ] 2026-06-15 10:01:15 - el_paso.save:44 - save finished in 0.532 seconds
[WARNING ] 2026-06-15 10:01:16 - el_paso.save:166 - Saving attempted, but product is missing some required variables for output flux!
[WARNING ] 2026-06-15 10:01:16 - el_paso.save:166 - Saving attempted, but product is missing some required variables for output alpha_and_energy!
[WARNING ] 2026-06-15 10:01:16 - el_paso.save:166 - Saving attempted, but product is missing some required variables for output mlt!
[WARNING ] 2026-06-15 10:01:16 - el_paso.save:166 - Saving attempted, but product is missing some required variables for output l

# Datasets

The saved datasets can be loaded using the `DataSet` class. We provide two concerete implmentation for the `DataSet`, namely `GFZDataSet` and `PRBEMDataSet`.

## GFZDataSet

`GFZDataSet/PRBEMDataSet` is used to load the data which is saved using `GFZStandard/PRBEMStandard`. The variables which can be accessed by each of these `DataSet` implementations are annotated in the respective classes in order to help user with the auto-completions.


In [10]:
from el_paso.dataset import GFZDataSet, PRBEMDataSet

mrb_gfz_data = GFZDataSet(mrb_gfz, start_time, end_time)
gfz_gfz_data = GFZDataSet(gfz_gfz, start_time, end_time)
mrb_prbem_data = PRBEMDataSet(mrb_prbem, start_time, end_time)
gfz_prbem_data = PRBEMDataSet(gfz_prbem, start_time, end_time)

[WARNING ] 2026-06-15 10:01:27 - el_paso.dataset.dataset_implementations:113 - Overriding `preferred_extension` to 'mat' since `GFZStrategy` is used, which only supports .mat files. Ignoring provided `preferred_extension` value.


In [11]:
mrb_prbem_data.metadata.datetime
mrb_prbem_data.metadata.to_dict()

[INFO    ] 2026-06-15 10:01:31 - el_paso.dataset.dataset:325 - Loading RBSP/mrb_prbem/RBSP/rbspa/rbspa_hope_20170701to20170731_T89.nc


{'Position': VariableMetadata(unit=Unit("km"), original_cadence_seconds=np.int64(0), source_files='rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf', description='Spacecraft position in geographic cartesian coordinates', processing_notes='', standard_name='Position'),
 'Epoch': VariableMetadata(unit=Unit("posixtime"), original_cadence_seconds=np.int64(0), source_files='rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf', description='Posix Time', processing_notes='', standard_name='Epoch'),
 'datetime': VariableMetadata(unit=Unit("posixtime"), original_cadence_seconds=np.int64(0), source_files=['r', 'b', 's', 'p', 'a', '_', 'r', 'e', 'l', '0', '4', '_', 'e', 'c', 't', '-', 'h', 'o', 'p', 'e', '-', 'p', 'a', '-', 'l', '3', '_', '2', '0', '1', '7', '0', '7', '1', '4', '_', 'v', '7', '.', '4', '.', '0', '.', 'c', 'd', 'f'], description='Python datetime objects converted from Epoch variable. This variable is not saved to disk but computed on the fly when requested.', processing_notes='1) Compute

In [12]:
mrb_prbem_data.Position.shape

(3940, 3)

In [13]:
mrb_gfz_data.metadata.datetime
mrb_gfz_data.metadata.to_dict()

[INFO    ] 2026-06-15 10:01:34 - el_paso.dataset.dataset:325 - Loading RBSP/mrb_gfz/RBSP/rbspa/rbspa_hope_20170701to20170731_T89.nc


{'xGEO': VariableMetadata(unit=Unit("RE"), original_cadence_seconds=np.int64(0), source_files='rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf', description='Position in geographic cartesian coordinates.', processing_notes='', standard_name='Position'),
 'time': VariableMetadata(unit=Unit("datenum"), original_cadence_seconds=np.int64(0), source_files='rbspa_rel04_ect-hope-pa-l3_20170714_v7.4.0.cdf', description='Time in MATLAB datenum format.', processing_notes='', standard_name='Epoch'),
 'datetime': VariableMetadata(unit=Unit("datenum"), original_cadence_seconds=np.int64(0), source_files=['r', 'b', 's', 'p', 'a', '_', 'r', 'e', 'l', '0', '4', '_', 'e', 'c', 't', '-', 'h', 'o', 'p', 'e', '-', 'p', 'a', '-', 'l', '3', '_', '2', '0', '1', '7', '0', '7', '1', '4', '_', 'v', '7', '.', '4', '.', '0', '.', 'c', 'd', 'f'], description='Python datetime objects converted from Epoch variable. This variable is not saved to disk but computed on the fly when requested.', processing_notes='1) Comput

In [14]:
mrb_gfz_data.xGEO.shape

(3940, 3)

In [15]:
gfz_gfz_data.metadata.datetime
gfz_gfz_data.metadata.to_dict()

[WARNING ] 2026-06-15 10:01:36 - el_paso.dataset.dataset:328 - Tried to load RBSP/gfz_gfz/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_flux_ver4.mat, but it does not exist


{}

In [16]:
gfz_gfz_data.xGEO.shape

[INFO    ] 2026-06-15 10:01:38 - el_paso.dataset.dataset:325 - Loading RBSP/gfz_gfz/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_xGEO_ver4.mat


(3940, 3)

In [17]:
gfz_prbem_data.metadata.datetime
gfz_prbem_data.metadata.to_dict()

[WARNING ] 2026-06-15 10:01:39 - el_paso.dataset.dataset:328 - Tried to load RBSP/gfz_prbem/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_flux_ver4.mat, but it does not exist


{}

In [18]:
gfz_prbem_data.Position.shape

[INFO    ] 2026-06-15 10:01:40 - el_paso.dataset.dataset:325 - Loading RBSP/gfz_prbem/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_xGEO_ver4.mat


(3940, 3)

# Setting data to `DataSet` externally

If in case, we want to attach some data to a `DataSet`, a complex `__setattr__` is implemented to achieve that.


In [19]:
from el_paso.dataset import DataSet

mock_strategy = ep.saving_strategies.MonthlyRBStrategy(
    base_data_path=".",
    mission="mock",
    satellite="mock",
    instrument="mock",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)
mock_ds = DataSet(mock_strategy, start_time, end_time)

In [20]:
mock_ds.time

[WARNING ] 2026-06-15 10:01:43 - el_paso.dataset.dataset:328 - Tried to load MOCK/mock/mock_mock_20170701to20170731_T89.nc, but it does not exist


array([], dtype=float64)

In [22]:
mock_ds.time = variables["Epoch"]
mock_ds.xGEO = variables["xGEO"]

Since the `mock_ds` is initialised using `GFZStandard`, only the variables which are associated with `GFZStandard` can be attributed to it. Setting an invalid variable will raise `AttributeError`. The same applies to `PRBEMStandard`


In [23]:
import traceback

try:
    mock_ds.Position = 1
except AttributeError as e:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_581244/4028079179.py", line 4, in <module>
    mock_ds.Position = 1
    ^^^^^^^^^^^^^^^^
  File "/home/bhaas/Repos/EL_PASO/bunch_download/el_paso/dataset/dataset.py", line 158, in __setattr__
    raise AttributeError(msg)
AttributeError: Cannot set attribute 'Position'. It is not part of the saving strategy: MonthlyRBStrategy(base_data_path=PosixPath('.'), mission='mock', satellite='mock', instrument='mock', mag_field='T89', data_standard=GFZStandard(), file_format='.nc').
